<a href="https://colab.research.google.com/github/NaveenHerath/llm-from-scratch/blob/main/day10_layer_norm_%26_dropout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Making the model safe to Scale Up
Layer Normalization and Dropout


In [14]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

batch_size = 32       # B: chunks per batch
block_size = 8        # T: the longest context the model can see
n_embd = 32           # C: numbers describing each character
n_head = 4            # heads per block
n_layer = 3           # how many blocks are stacked
dropout = 0.2         # new: fraction of numbers switched off in training
max_iters = 5000
eval_interval = 500
eval_iters = 200
learning_rate = 1e-3

In [15]:
!wget -q -nc https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

In [16]:
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

In [17]:
class Head(nn.Module):
    """ one head of masked self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)                    # new

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)                                       # (B, T, hs)
        q = self.query(x)                                     # (B, T, hs)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5     # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)                               # new: drop some attention weights
        v = self.value(x)                                     # (B, T, hs)
        return wei @ v                                        # (B, T, hs)

In [18]:
class MultiHeadAttention(nn.Module):
    """ several heads of self-attention side by side """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)                    # new

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)   # (B, T, 32)
        out = self.dropout(self.proj(out))                    # new: dropout after proj
        return out

In [19]:
class FeedForward(nn.Module):
    """ a small network applied to each position on its own """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),                              # new
        )

    def forward(self, x):
        return self.net(x)

In [20]:
class Block(nn.Module):
    """ one transformer block, pre-norm """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)                       # new
        self.ln2 = nn.LayerNorm(n_embd)                       # new

    def forward(self, x):
        x = x + self.sa(self.ln1(x))      # normalise, attend, add back
        x = x + self.ffwd(self.ln2(x))    # normalise, compute, add back
        return x

In [21]:
#tiny gpt with a final layer norm
class TinyGPT(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)                            # new: final LayerNorm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)                   # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T))    # (T, C)
        x = tok_emb + pos_emb                                       # (B, T, C)
        x = self.blocks(x)                                          # (B, T, C)
        x = self.ln_f(x)                                            # (B, T, C)
        logits = self.lm_head(x)                                    # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [22]:
def train(model):
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    history, step_losses = [], []
    for step in range(max_iters + 1):
        if step % eval_interval == 0:
            l = estimate_loss(model)
            history.append((step, l['train'], l['val']))
            print(f"step {step:4d}: train {l['train']:.4f}, val {l['val']:.4f}")
        if step == max_iters:
            break
        xb, yb = get_batch('train')
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        step_losses.append(loss.item())
    return history, step_losses

In [23]:
torch.manual_seed(1337)
model = TinyGPT()
print(sum(p.numel() for p in model.parameters()), 'parameters')
history, step_losses = train(model)

42369 parameters
step    0: train 4.3103, val 4.3100
step  500: train 2.4437, val 2.4468
step 1000: train 2.3216, val 2.3296
step 1500: train 2.2482, val 2.2717
step 2000: train 2.2128, val 2.2182
step 2500: train 2.1815, val 2.2022
step 3000: train 2.1543, val 2.1786
step 3500: train 2.1343, val 2.1743
step 4000: train 2.1187, val 2.1635
step 4500: train 2.0942, val 2.1386
step 5000: train 2.0895, val 2.1223


In [24]:
x = torch.randn(1, 8, n_embd) * 3 + 5        # numbers with mean ~5, spread ~3
ln = model.blocks[0].ln1

with torch.no_grad():
    token = x[0, 2]                                         # one token's 32 numbers
    print('before:      mean', round(token.mean().item(), 3), ' spread', round(token.std(unbiased=False).item(), 3))
    normed = F.layer_norm(x, (n_embd,))[0, 2]                # the normalising step alone
    print('normalised:  mean', round(normed.mean().item(), 3), ' spread', round(normed.std(unbiased=False).item(), 3))
    after = ln(x)[0, 2]                                      # with the learned scale and shift
    print('after ln1:   mean', round(after.mean().item(), 3), ' spread', round(after.std(unbiased=False).item(), 3))

before:      mean 5.422  spread 2.516
normalised:  mean 0.0  spread 1.0
after ln1:   mean -0.005  spread 1.074


In [26]:
torch.manual_seed(42)
model.eval()
start = torch.zeros((1, 1), dtype=torch.long)
print(decode(model.generate(start, max_new_tokens=300)[0].tolist()))
model.train();



KIDYOLIONCES:
Why wo ky chat no, on annd bed fo thou fore heed lake the dreacty wethall kungnl herir, that in'n
ceasied famensem:
It hing thall colve whought for not prenten if Gichs both.

POMENENIO:
Wen coured hing upporn;
Noung,
Wellurd
Therit my diff thou grown
seases.

DUKINNARCIS: The ancs br
